# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIRˆ² dataset (Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya) using the `mlcroissant` library.

### Dataset Source
The dataset is described using a Croissant schema accessible by a public URL.

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access and display metadata (no subscripting, access attributes)
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs. Using the Croissant schema, we can explore the dataset's structure programmatically using `mlcroissant`.

We will enumerate all record sets, their `@id`, their fields, and field `@id`s.

In [ ]:
# List record sets and fields using mlcroissant's metadata graph
record_sets = list(dataset.metadata.record_sets)

print(f"Found {len(record_sets)} record set(s) in the dataset:")
for rec_set in record_sets:
    print(f"\nRecord set: {rec_set.name}\n  @id: {rec_set.id}")
    print("  Fields:")
    for field in rec_set.fields:
        print(f"    - {field.name} (@id: {field.id}) [dataType: {getattr(field, 'data_type', '-')}]")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All record set IDs and field IDs (from the previous section) are used here. You can select which record set(s) to analyze further by referencing their `@id`.

In [ ]:
# Extract data from all record sets, using @id as the key
dataframes = {}
all_columns_by_rs = {}

for rec_set in record_sets:
    rec_id = rec_set.id
    print(f"Loading records from record set: {rec_set.name} (@id: {rec_id}) ...")
    records_list = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records_list)
    dataframes[rec_id] = df
    all_columns_by_rs[rec_id] = df.columns.tolist()
    print(f"  Columns: {df.columns.tolist()}")
    if not df.empty:
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping by key attributes.

Below, choose a record set (`@id`) and a numeric field (`@id`, i.e., a column name in the DataFrame) for demonstration. Adjust as appropriate based on the columns listed above.

In [ ]:
# Example: pick the main record set and a numeric field.
# Let's select the first record set by default. Adjust as needed.
if len(dataframes) == 0:
    print("No record sets found to analyze.")
else:
    # Select record set and confirm fields
    selected_rec_id = list(dataframes.keys())[0]
    df = dataframes[selected_rec_id]
    print(f"Selected record set: {selected_rec_id}")
    print(f"Available columns: {df.columns.tolist()}")

    # Attempt to auto-pick a likely numeric field
    numeric_candidates = [col for col in df.columns if df[col].dtype in [float, int, 'float64', 'int64']]
    if not numeric_candidates:
        # Try to infer numeric-looking columns if object type present
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(pd.to_numeric(df[col], errors='coerce')):
                numeric_candidates.append(col)
        if numeric_candidates:
            numeric_field = numeric_candidates[0]
            df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile as a dynamic threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold:.2f} (75th percentile):")
        display(filtered_df.head())
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Try grouping by a categorical column (not the numeric)
        group_candidates = [col for col in df.columns if col != numeric_field and df[col].dtype == object]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"Grouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped_df)
    else:
        print("No numeric column detected in selected record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example visualization if a numeric field exists. You may customize this cell according to the fields available in your dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) == 0:
    print("No data available for visualization.")
elif 'numeric_field' in locals() and numeric_field in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Optional: If grouping variable exists, show boxplot
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable numeric field found for visualization.")

## 6. Conclusion
In this notebook, we leveraged the `mlcroissant` library to load the Croissant-described dataset, enumerate its record sets and fields by `@id`, extracted the actual survey data programmatically, and performed basic exploratory and statistical analysis based on its structure. This approach ensures reproducibility, transparent referencing, and easy adaptation for further, domain-specific analytics workflows.